3 — Testing & Validation\nWalk-forward + backtest portefeuille FTMO + stress slippage sur `dataset.parquet`. Produit un verdict **GO / NO-GO**.\n\nOrdre réel d'exécution : **NB1 → NB3 (validation) → NB2 (training ONNX)** — on ne fige le modèle final qu'après un GO.

In [18]:
# pip install scikit-learn pandas numpy pyarrow

In [19]:
# === CONFIG PARTAGEE ===
SYMBOLS  = ['EURJPY','USDJPY','GBPJPY','GBPUSD','EURUSD','NZDUSD','AUDJPY','AUDUSD']
SL_PIPS  = {'AUDJPY':38,'AUDUSD':28,'EURJPY':48,'EURUSD':32,
            'GBPJPY':59,'GBPUSD':42,'NZDUSD':28,'USDJPY':39}
SPREAD_PIPS = {'AUDJPY':0.7,'AUDUSD':0.4,'EURJPY':0.6,'EURUSD':0.2,
               'GBPJPY':0.8,'GBPUSD':0.4,'NZDUSD':0.5,'USDJPY':0.3}
RR=1.70; MAX_HOLD_H=168; YEARS_BACK=10; SELECT_Q=0.95
FEAT=['rsi14','pctB','bw','ext','adx14','atr_norm','body','up_wick','dn_wick','bull',
      'dist_hi20','dist_lo20','bars_since_flip','ret5_atr','d1_state','d1_rsi',
      'd1_aligned','hour','dow','dir_b','sym_c']
SYM_CODE={s:i for i,s in enumerate(sorted(SYMBOLS))}
DATASET='dataset.parquet'


In [20]:
# === CHARGEMENT ===
import pandas as pd, numpy as np
d=pd.read_parquet(DATASET).dropna(subset=['R_net']).copy()
d['bars_since_flip']=d['bars_since_flip'].clip(upper=300)  # parite EA (garde-fou si vieux dataset)
d['year']=pd.to_datetime(d.time).dt.year
print(len(d),'lignes |',d.year.min(),'->',d.year.max())
print('R_net moyen univers: {:+.4f}'.format(d.R_net.mean()))

317134 lignes | 2001 -> 2026
R_net moyen univers: -0.0347


In [21]:
# === BACKTEST PORTEFEUILLE (contraintes FTMO) ===
def portfolio(sel,rcol='R_net',risk=0.0035):
    sel=sel.sort_values('entry_time')
    eq=0.;peak=0.;mdd=0.;day=None;dpl=0.;busy={};slots=[];n=0;wins=0
    for _,r in sel.iterrows():
        t=pd.Timestamp(r.entry_time)
        if t.date()!=day: day=t.date();dpl=0.
        slots=[x for x in slots if x>t]
        if dpl<=-0.04 or (peak-eq)>=0.10: continue          # stop jour / DD max
        if busy.get(r.symbol) and busy[r.symbol]>t: continue # 1 pos/symbole
        if len(slots)>=4: continue                           # 4 simultanees max
        pl=r[rcol]*risk; eq+=pl;dpl+=pl;n+=1;wins+=(r[rcol]>0)
        peak=max(peak,eq);mdd=max(mdd,peak-eq)
        end=t+pd.Timedelta(hours=48);busy[r.symbol]=end;slots.append(end)
    return n,eq,mdd,wins/max(n,1)

In [22]:
# === WALK-FORWARD (seuil appris sur train uniquement) ===
from sklearn.ensemble import HistGradientBoostingRegressor
results=[]; ok_windows=0
for tr_end,te_lo,te_hi in [(2018,2019,2021),(2021,2022,2024)]:
    tr=d[d.year<=tr_end]; te=d[(d.year>=te_lo)&(d.year<=te_hi)].copy()
    if len(tr)<5000 or len(te)<5000:
        print(f'fenetre {te_lo}-{te_hi}: donnees insuffisantes'); continue
    m=HistGradientBoostingRegressor(max_iter=300,max_depth=6,learning_rate=0.06,
        l2_regularization=1.0,random_state=0)
    m.fit(tr[FEAT],tr['R_net'])
    thr=np.quantile(m.predict(tr[FEAT]),SELECT_Q)
    te['pred']=m.predict(te[FEAT])
    sel=te[te.pred>=thr]
    print(f'=== test {te_lo}-{te_hi} : {len(sel)} signaux (seuil train {thr:+.4f}) ===')
    # detail annuel
    for y,g in sel.groupby('year'):
        n,eq,mdd,wr=portfolio(g)
        print(f'  {y}: trades={n:3d} | gain={eq:+6.1%} | maxDD={mdd:4.1%} | win={wr:.0%}')
    n,eq,mdd,wr=portfolio(sel)
    # stress slippage 1 pip / cote
    slip=(2*1.0)/sel.symbol.map(SL_PIPS)
    r_slip=(sel.R_net-slip).mean()
    print(f'  TOTAL: trades={n} gain={eq:+.1%} maxDD={mdd:.1%} win={wr:.0%}')
    print(f'  R_net moyen sel: {sel.R_net.mean():+.3f} | avec slippage 1pip/cote: {r_slip:+.3f}')
    good = eq>0 and mdd<0.08 and r_slip>0.05
    print(f'  fenetre {"VALIDE" if good else "REJETEE"}')
    results.append((f'{te_lo}-{te_hi}',eq,mdd,r_slip,good)); ok_windows+=int(good)

=== test 2019-2021 : 2356 signaux (seuil train +0.1676) ===
  2019: trades=288 | gain=+24.8% | maxDD=5.0% | win=49%
  2020: trades=239 | gain= +2.3% | maxDD=9.2% | win=40%
  2021: trades=140 | gain= +1.6% | maxDD=10.1% | win=40%
  TOTAL: trades=667 gain=+25.1% maxDD=10.3% win=43%
  R_net moyen sel: +0.025 | avec slippage 1pip/cote: -0.028
  fenetre REJETEE
=== test 2022-2024 : 1827 signaux (seuil train +0.1573) ===
  2022: trades=231 | gain= +6.6% | maxDD=5.3% | win=41%
  2023: trades=206 | gain= +4.3% | maxDD=8.8% | win=40%
  2024: trades=127 | gain= -9.1% | maxDD=10.0% | win=31%
  TOTAL: trades=241 gain=+2.1% maxDD=10.0% win=39%
  R_net moyen sel: -0.040 | avec slippage 1pip/cote: -0.093
  fenetre REJETEE


In [23]:
# === VERDICT ===
verdict='GO' if ok_windows==2 else 'NO-GO'
print('='*50)
print(f'VERDICT: {verdict}  ({ok_windows}/2 fenetres valides)')
print('='*50)
if verdict=='GO':
    print('-> Tu peux lancer le notebook 2 (Training ONNX).')
else:
    print('-> NE PAS entrainer ni trader. Partager dataset.parquet pour analyse.')

VERDICT: NO-GO  (0/2 fenetres valides)
-> NE PAS entrainer ni trader. Partager dataset.parquet pour analyse.
